In [32]:
from z3 import *
from abc import ABC, abstractmethod

In [22]:
class LanguageE:
    def __init__(self):
        # Core fields required by the engine
        self.axioms: list[ExprRef] = []
        self.solver: Solver = Solver()
        
        # Declare uninterpreted sorts for Euclid's elements
        self.PointSort: SortRef = DeclareSort('Point')
        self.LineSort: SortRef = DeclareSort('Line')
        self.CircleSort: SortRef = DeclareSort('Circle')
        
        # Constants
        self.RightAngle: ExprRef = Real('RightAngle')
        
        # --- Predicates / Relations (Return BoolSort) ---
        # Function('Name', InputSort1, InputSort2, ..., OutputSort)
        self.Between = Function('Between', self.PointSort, self.PointSort, self.PointSort, BoolSort())
        self.OnLine = Function('OnLine', self.PointSort, self.LineSort, BoolSort())
        self.OnCircle = Function('OnCircle', self.PointSort, self.CircleSort, BoolSort())
        self.Inside = Function('Inside', self.PointSort, self.CircleSort, BoolSort())
        self.Center = Function('Center', self.PointSort, self.CircleSort, BoolSort())
        self.SameSide = Function('SameSide', self.PointSort, self.PointSort, self.LineSort, BoolSort())
        
        # Intersection Predicates
        self.Intersectsll = Function('Intersectsll', self.LineSort, self.LineSort, BoolSort())
        self.Intersectslc = Function('Intersectslc', self.LineSort, self.CircleSort, BoolSort())
        self.Intersectscc = Function('Intersectscc', self.CircleSort, self.CircleSort, BoolSort())
        
        # --- Magnitudes / Metrics (Return RealSort) ---
        self.Segment = Function('Segment', self.PointSort, self.PointSort, RealSort())
        self.Angle = Function('Angle', self.PointSort, self.PointSort, self.PointSort, RealSort())
        self.Area = Function('Area', self.PointSort, self.PointSort, self.PointSort, RealSort())

In [31]:
class Proof:
    def __init__(self):
        self.points: set[Point] = set()
        self.lines: set[Line] = set()
        self.circles: set[Circle] = set()
        
        self.assumptions: set[ExprRef] = set()
        self.conclusions: set[ExprRef] = set()
        self.isProved: bool = False

        self.language: LanguageE = LanguageE()
        self.solver = self.language.solver
        
    def construct(self, constructableObject):
        pass
    
    def hence(self, exprRef):
        pass
    
    def asserts(self, proof):
        pass
    
    def qed(self):
        pass

In [34]:
class ConstructableObject(ABC):
    def __init__(self, label: str, sort: str):
        # Fields straight from the rojas_thesis.pdf object model:
        self.z3Expr: ExprRef = None         # The underlying Z3 expression representation
        self.label: str = label             # e.g., "a", "b", "L"
        self.sort: str = sort               # "Point", "Line", or "Circle"[cite: 1]
        self.prereqs: list[ExprRef] = []    # Language E preconditions for construction[cite: 1]
        self.conclusions: list[ExprRef] = []# Language E post-conditions/conclusions[cite: 1]
        self.isDistinct: bool = True        # Unique object flag[cite: 1]

    @abstractmethod
    def equals(self, obj: object) -> bool: # Maps to + equals(Obj) in the UML[cite: 1]
        """Every geometric object must implement its own equality logic."""
        pass

In [37]:
class Circle(ConstructableObject):
    def __init__(self, label: str, language: LanguageE):
        # 1. Initialize the abstract parent fields with the sort "Circle"
        super().__init__(label, "Circle")
        
        # 2. Create the actual Z3 constant object using the CircleSort we defined
        self.z3Expr = Const(label, language.CircleSort)
        
    def centerThrough(self, point1: 'Point', point2: 'Point', language: LanguageE):
        """
        Defines a circle with center at point1, passing through point2.
        """
        # Prerequisites: To draw a valid circle, the center and radius point cannot be identical[cite: 1]
        self.prereqs = [point1.z3Expr != point2.z3Expr]
        
        # Conclusions: Drawing this circle introduces two facts into Language E[cite: 1]:
        # 1. point1 is the center of this circle[cite: 1]
        # 2. point2 lies on the boundary of this circle[cite: 1]
        self.conclusions = [
            language.Center(point1.z3Expr, self.z3Expr),
            language.OnCircle(point2.z3Expr, self.z3Expr)
        ]
        
    def equals(self, obj: object) -> bool:
        # Fulfilling the abstract method obligation[cite: 1]
        if isinstance(obj, Circle):
            return self.label == obj.label
        return False

In [38]:
class Line(ConstructableObject):
    def __init__(self, label: str, language: LanguageE):
        # 1. Initialize the abstract parent fields with the sort "Line"
        super().__init__(label, "Line")
        
        # 2. Create the actual Z3 constant object using the LineSort defined in LanguageE
        self.z3Expr = Const(label, language.LineSort)
        
    def through(self, point1: 'Point', point2: 'Point', language: LanguageE):
        """
        Defines a straight line that passes through two distinct points.
        """
        # Prerequisites: To construct a unique line, the two points must not be identical
        self.prereqs = [point1.z3Expr != point2.z3Expr]
        
        # Conclusions: Drawing this line adds two facts to the proof engine:
        # 1. point1 is on this line
        # 2. point2 is on this line[cite: 1]
        self.conclusions = [
            language.OnLine(point1.z3Expr, self.z3Expr),
            language.OnLine(point2.z3Expr, self.z3Expr)
        ]
        
    def equals(self, obj: object) -> bool:
        # Fulfilling the abstract method obligation from ConstructableObject[cite: 1]
        if isinstance(obj, Line):
            return self.label == obj.label
        return False

In [39]:
from z3 import Const

class Point(ConstructableObject):
    def __init__(self, label: str, language: LanguageE):
        # 1. Initialize abstract parent fields with the sort "Point"
        super().__init__(label, "Point")
        
        # 2. Create the Z3 constant object using PointSort from LanguageE
        self.z3Expr = Const(label, language.PointSort)
        
    def onLine(self, line: 'Line', language: LanguageE):
        """Constructs an arbitrary point on a given line."""
        self.prereqs = []
        self.conclusions = [language.OnLine(self.z3Expr, line.z3Expr)]
        
    def onCircle(self, circle: 'Circle', language: LanguageE):
        """Constructs an arbitrary point on the boundary of a circle.[cite: 1]"""
        self.prereqs = []
        self.conclusions = [language.OnCircle(self.z3Expr, circle.z3Expr)]
        
    def between(self, point1: 'Point', point2: 'Point', language: LanguageE):
        """Constructs a point collinear to and strictly between two other points.[cite: 1]"""
        # Prerequisite: The two bounding points cannot be identical[cite: 1]
        self.prereqs = [point1.z3Expr != point2.z3Expr]
        self.conclusions = [language.Between(point1.z3Expr, self.z3Expr, point2.z3Expr)]
        
    def sameside(self, point1: 'Point', line: 'Line', language: LanguageE):
        """Constructs a point on the same side of a line as another given point.[cite: 1]"""
        self.prereqs = []
        self.conclusions = [language.SameSide(self.z3Expr, point1.z3Expr, line.z3Expr)]
        
    def opposite(self, point1: 'Point', line: 'Line', language: LanguageE):
        """Constructs a point on the opposite side of a line from a given point.[cite: 1]"""
        self.prereqs = []
        # In Language E, opposite side is logically the negation of SameSide[cite: 1]
        from z3 import Not
        self.conclusions = [Not(language.SameSide(self.z3Expr, point1.z3Expr, line.z3Expr))]
        
    def inside(self, circle: 'Circle', language: LanguageE):
        """Constructs a point located inside the interior area of a circle.[cite: 1]"""
        self.prereqs = []
        self.conclusions = [language.Inside(self.z3Expr, circle.z3Expr)]
        
    def outside(self, circle: 'Circle', language: LanguageE):
        """Constructs a point located strictly outside a circle.[cite: 1]"""
        self.prereqs = []
        from z3 import Not
        self.conclusions = [
            Not(language.Inside(self.z3Expr, circle.z3Expr)),
            Not(language.OnCircle(self.z3Expr, circle.z3Expr))
        ]
        
    def intersectsLines(self, line1: 'Line', line2: 'Line', language: LanguageE):
        """Constructs a point where two lines cross each other.[cite: 1]"""
        # Prerequisite: The two lines must actually intersect[cite: 1]
        self.prereqs = [language.Intersectsll(line1.z3Expr, line2.z3Expr)]
        self.conclusions = [
            language.OnLine(self.z3Expr, line1.z3Expr),
            language.OnLine(self.z3Expr, line2.z3Expr)
        ]
        
    def intersectsCircleLine(self, circle: 'Circle', line: 'Line', language: LanguageE):
        """Constructs a point where a circle boundary intersects a straight line.[cite: 1]"""
        # Prerequisite: The line and circle must intersect[cite: 1]
        self.prereqs = [language.Intersectslc(line.z3Expr, circle.z3Expr)]
        self.conclusions = [
            language.OnCircle(self.z3Expr, circle.z3Expr),
            language.OnLine(self.z3Expr, line.z3Expr)
        ]
        
    def intersectsCircleCircle(self, circle1: 'Circle', circle2: 'Circle', language: LanguageE):
        """Constructs a point at the intersection point of two circle boundaries.[cite: 1]"""
        # Prerequisite: Both circles must intersect[cite: 1]
        self.prereqs = [language.Intersectscc(circle1.z3Expr, circle2.z3Expr)]
        self.conclusions = [
            language.OnCircle(self.z3Expr, circle1.z3Expr),
            language.OnCircle(self.z3Expr, circle2.z3Expr)
        ]
        
    def equals(self, obj: object) -> bool:
        # Fulfilling the abstract method obligation[cite: 1]
        if isinstance(obj, Point):
            return self.label == obj.label
        return False